In [ ]:
from pathlib import Path

import torch
from e2_tts_pytorch import E2TTS, DurationPredictor

from torch.optim import Adam

from e2_tts_pytorch.trainer import (
    TextAudioDataset,
    E2Trainer
)

In [ ]:
!wandb login 8b4188b5e205e983805aebdbcb8b22fe8db630a4

In [ ]:

# duration_predictor = DurationPredictor(
#     transformer = dict(
#         dim = 80,
#         depth = 2,
#     )
# )

e2tts = E2TTS(
    # duration_predictor = duration_predictor,
    cond_drop_prob = 0.0,
    transformer = dict(
        dim = 128,
        depth = 4,
        heads = 8,
        max_seq_len = 2048,
        num_gateloop_layers = 0,
        skip_connect_type = 'concat'
    ),
    mel_spec_kwargs = dict(
        filter_length = 1024,
        hop_length = 256,
        win_length = 1024,
        n_mel_channels = 100,
        sampling_rate = 24000,
    ),
    frac_lengths_mask = (0.5, 0.7)
)

trainable_param_count = sum(p.numel() for p in e2tts.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_param_count}")

In [ ]:
train_dataset = TextAudioDataset(Path("~/data/LibriTTS_R/dev-clean").expanduser())

optimizer = Adam(e2tts.parameters(), lr = 1e-4)

trainer = E2Trainer(
    e2tts,
    optimizer,
    num_warmup_steps = 1000,
    checkpoint_path = 'e2tts.pt',
    log_file = 'e2tts.txt',
    # accelerate_kwargs = {"mixed_precision": "fp16"}
)

epochs = 50
batch_size = 4
grad_accumulation_steps = 1

print("Training...")

trainer.train(train_dataset, epochs, batch_size, grad_accumulation_steps, save_step = 1000)

torch.save(e2tts, "e2tts_test.pt")